# Measuring fan blade width with a laser and a photodiode

This notebook puts the ideas from `nidaqmx_tutorial.ipynb` into a small experiment.
A laser beam passes through a spinning fan onto a photodiode. Each time a blade passes,
it blocks the beam and the photodiode signal drops. From the recorded signal we measure:

* the **pulse width** τ, i.e. how long each blade blocks the beam
* the **blade period** T, i.e. the time between one blade and the next

From these, together with the number of blades and where the beam crosses the fan, we get the blade width.

```
          laser ───────────────► ┃ ◄─ fan blade ─► photodiode ──► AI0
                                  (beam at radius r from the hub)

 photodiode   ─────┐     ┌─────────┐     ┌─────────┐     ┌────
 signal            │     │         │     │         │     │
                   └─────┘         └─────┘         └─────┘
                   ◄─ τ ─►
                   ◄────── T ──────►
```

The fan and the laser run from their **own power supplies**, at a fixed speed and
brightness. The DAQ only records: the photodiode is the only thing connected to it.

### The physics

The fan has `N` blades, so it turns once every `N·T` seconds. Its rotation frequency is

$$f = \frac{1}{N\,T}$$

At radius $r$ the blade moves at $v = 2\pi r f$, so the distance swept while the beam is blocked is

$$w_\text{measured} = v\,\tau = \frac{2\pi r\,\tau}{N\,T}$$

The beam has a finite diameter $d$. It starts being blocked when the blade's leading edge
reaches the near side of the beam, and stops when the trailing edge leaves the far side, so

$$w_\text{blade} \approx w_\text{measured} - d$$

Note that $w$ depends only on the **ratio** τ/T. We don't need to know the fan's speed in
advance, because T measures it from the same recording. So small changes in fan speed
between recordings shouldn't change the result. Section 5 checks this, and also checks
that the sample rate is high enough.

### Which tutorial ideas are used where

| Step | Tutorial idea |
|---|---|
| Live view for aligning the beam | Continuous AI loop / live plot (4a, 4b) |
| Measurement | Finite, hardware-timed AI (1b) |
| Checking the sample rate | Choosing `RATE` for hardware-timed AI (1b) |
| Analysis | numpy on a finite AI recording (1b) |

### Wiring and safety

| Signal | DAQ pin | Notes |
|---|---|---|
| Photodiode | `AI0` (RSE) | Use a photodiode amplifier, or a photodiode with a load resistor, so you get a voltage. Ground goes to AI GND. |
| Fan | not connected | Runs from its own power supply at a fixed speed. |
| Laser | not connected | Runs from its own power supply. |

No analog outputs are used in this notebook. **If the AO0 → AI0 loopback wire from the
exercises is still connected, remove it.** Otherwise AI0 measures the AO output instead of
your photodiode.

⚠️ **Laser safety:** never look into the beam, and watch out for reflections off the fan blades.
Use a low-power module (class 2 or below) and keep the beam path below eye level.
⚠️ Keep fingers away from the spinning fan. It keeps spinning when your code stops,
so switch it off at its own supply before reaching near it.

---
## Why a fan? The neuroscience connection

This setup is a small, safe model of two instruments that neuroscience labs use every day.

### A scanning (multiphoton) microscope

A two-photon microscope has **no camera**. It focuses a pulsed laser to a single point, moves
that point across the tissue with scan mirrors, and records the fluorescence from each
position with a single detector, a photomultiplier tube (PMT). The image only exists
because we know **where the beam was at each moment**: time is converted into position.

Our fan setup does the same thing the other way round. The beam stays still and the object
moves through it. We record the photodiode over time and use the known speed to turn time into distance:

$$\text{size} = \text{speed} \times \text{time}$$

That is exactly how a microscope's pixel size is set: µm per pixel = scan speed × time per pixel.
It's also why scan speed has to be known precisely. A **resonant scanner** (typically ~8 kHz)
moves sinusoidally, so it travels fastest in the middle of each line and slows at the edges,
and imaging software has to correct for that or the image is distorted at the edges. Measuring the fan's
rotation from the blade period T is like measuring a resonant scanner's actual frequency
from its sync output.

### An electrophysiology rig

Look at the photodiode trace and think "electrode": a steady baseline interrupted by brief,
stereotyped events. Finding the blade pulses is the same problem as **spike detection**:

* set a threshold relative to the baseline and the noise
* find the threshold crossings
* measure each event's **width** and the **interval** between events

The photodiode itself is standard kit in **optogenetics**. A photodiode picking off some of
the stimulation light is how labs check that the light came on when it was commanded, and
for as long. Here the laser runs from its own supply, so we only use the measuring side of that.

### How the parts map

| This experiment | Two-photon microscope | Ephys / optogenetics rig |
|---|---|---|
| Laser beam | Excitation laser | Optogenetic laser/LED |
| Fan blades moving through the beam | Scan mirrors moving the beam over the tissue | — |
| Photodiode on AI0 | PMT detecting fluorescence | Electrode + amplifier, or a light-monitoring photodiode |
| Dark level (blade fully blocking the beam) | PMT offset, subtracted from images | Baseline |
| Blade pulse | Beam passing over a labelled cell | A spike |
| Pulse width τ → blade width | Time per pixel → pixel size | Spike width |
| Blade period T | Line period of the scanner | Inter-spike interval → firing rate |
| Hardware sample clock: sample *i* is at *i* / `RATE` | Pixel clock: sample *i* is pixel *i* | The amplifier's fixed sampling rate |

The 🧠 notes through the notebook point out where each step shows up in a real experiment.

---
## 0. Setup

Leave `SIMULATE = True` to run the whole notebook without hardware. It generates a
realistic fake photodiode signal, so you can check the analysis first.
Set it to `False` once everything is wired up.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

import nidaqmx
from nidaqmx.constants import AcquisitionType, TerminalConfiguration

SIMULATE = True   # <-- set False to use the real DAQ

### Configuration

Everything you may need to change is in this one cell.

In [ ]:
# --- DAQ channel ---
DEV = "Dev1"           # device name, as shown in NI MAX
PD_CHAN = f"{DEV}/ai0"

# --- Acquisition ---
RATE = 20_000          # samples/s. Check your device's maximum (e.g. USB-6001 is 20 kS/s)
DURATION = 1.0         # s per recording

# --- The fan and the optics (measure these!) ---
N_BLADES = 7
BEAM_RADIUS_MM = 30.0  # distance from the fan's axis to the laser beam
BEAM_DIAMETER_MM = 1.0 # laser spot size at the fan

# --- Analysis ---
MIN_PULSE_S = 200e-6   # ignore "pulses" shorter than this (noise near the threshold)

### Fake data for `SIMULATE = True`

This models a fan spinning at about 20 revolutions per second, with blades 15 mm wide and a
little noise. Like a real fan on its own supply, its speed varies by a few percent from one
recording to the next. It has nothing to do with DAQmx. It's only there so the analysis can
be tested against a known answer.

In [ ]:
SIM_TRUE_BLADE_MM = 15.0
SIM_FAN_HZ = 20.0          # rotation frequency (revolutions per second)
SIM_SPEED_JITTER = 0.03    # 3 % speed variation between recordings
SIM_DARK_V, SIM_LIGHT_V = 0.05, 2.0
rng = np.random.default_rng(0)


def fake_photodiode(n, rate):
    """n samples of photodiode signal at `rate` samples/s."""
    t = np.arange(n) / rate
    fan_hz = SIM_FAN_HZ * (1 + rng.normal(0, SIM_SPEED_JITTER))
    angle = 2 * np.pi * fan_hz * t + rng.uniform(0, 2 * np.pi)
    blade_pitch = 2 * np.pi / N_BLADES
    blocked_angle = (SIM_TRUE_BLADE_MM + BEAM_DIAMETER_MM) / BEAM_RADIUS_MM
    blocked = (angle % blade_pitch) < blocked_angle
    light = np.where(blocked, SIM_DARK_V, SIM_LIGHT_V)
    return light + rng.normal(0, 0.02, n)

---
## 1. Checking the signal: live view (continuous AI)

Switch the laser and the fan on at their own power supplies. While adjusting the laser and
photodiode, it helps to watch the signal live. This is the continuous loop from tutorial
section 4b.

* If you can switch the fan off, first align for the **highest** signal with the beam unblocked.
* With the fan spinning, you should see a regular train of dips. The top of the trace should
  stay high, and the dips should go close to zero when a blade fully blocks the beam.

In [ ]:
from IPython.display import clear_output, display


def live_view(seconds=10.0, chunk=5_000):
    """Plot the photodiode signal live for a while (continuous AI)."""
    if SIMULATE:
        print("[sim] live view needs hardware; skipping")
        return
    fig, ax = plt.subplots(figsize=(10, 3))
    with nidaqmx.Task() as task:
        task.ai_channels.add_ai_voltage_chan(
            PD_CHAN, terminal_config=TerminalConfiguration.RSE, min_val=-10, max_val=10
        )
        task.timing.cfg_samp_clk_timing(
            rate=RATE, sample_mode=AcquisitionType.CONTINUOUS, samps_per_chan=10 * chunk
        )
        task.start()
        t_end = time.time() + seconds
        while time.time() < t_end:
            data = np.array(task.read(number_of_samples_per_channel=chunk))
            ax.clear()
            ax.plot(np.arange(chunk) / RATE * 1000, data)
            ax.set_ylim(-0.5, 10)
            ax.set_xlabel("Time (ms)")
            ax.set_ylabel("Photodiode (V)")
            ax.set_title(f"Mean {data.mean():.3f} V")
            clear_output(wait=True)
            display(fig)
    plt.close(fig)


live_view(seconds=10)

---
## 2. The measurement: finite, hardware-timed AI

One measurement is a finite AI recording of `DURATION` seconds at `RATE` samples/s
(tutorial section 1b). The DAQ's own clock times every sample, so sample *i* was taken at
exactly *i* / `RATE` seconds. That's what lets us turn sample counts into times.

We don't need a trigger. The fan spins all the time, and it doesn't matter where in a
rotation the recording starts. (Exercise 03 needed a trigger because an output and a
recording had to line up. Here there's no output.)

> **🧠 In the lab:** this is a *free-running* recording, like recording spontaneous activity
> in ephys, as opposed to sweeps triggered by a stimulus. What makes it usable is the
> hardware sample clock: in a microscope, it's what guarantees that PMT sample *i* is pixel *i*.
> A loop of on-demand reads timed by Python (tutorial 1a) would give irregular spacing, and
> every width and period computed from it would be wrong.

In [ ]:
def record(rate=RATE, duration=DURATION):
    """One finite, hardware-timed photodiode recording. Returns (t, photodiode)."""
    n = int(rate * duration)
    t = np.arange(n) / rate

    if SIMULATE:
        return t, fake_photodiode(n, rate)

    with nidaqmx.Task() as task:
        task.ai_channels.add_ai_voltage_chan(
            PD_CHAN, terminal_config=TerminalConfiguration.RSE, min_val=-10, max_val=10
        )
        task.timing.cfg_samp_clk_timing(
            rate=rate, sample_mode=AcquisitionType.FINITE, samps_per_chan=n
        )
        pd = np.array(task.read(number_of_samples_per_channel=n, timeout=duration + 5))

    return t, pd

In [ ]:
t, pd = record()

fig, axes = plt.subplots(2, 1, figsize=(10, 5))
axes[0].plot(t, pd, lw=0.5)
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Photodiode (V)")
axes[0].set_title("Whole recording")

zoom = t < 0.1
axes[1].plot(t[zoom] * 1000, pd[zoom])
axes[1].set_xlabel("Time (ms)")
axes[1].set_ylabel("Photodiode (V)")
axes[1].set_title("Zoom: first 100 ms")
plt.tight_layout()
plt.show()

---
## 3. Finding the pulses

1. **Dark level:** the signal while a blade fully blocks the beam. We use the 5th percentile
   of the recording.
2. **Light level:** the unblocked signal. We use the 95th percentile, so a few noisy samples don't skew it.
3. **Threshold:** halfway between the two. Below the threshold means *blocked*.
4. **Edges:** where *blocked* changes from False to True (blade arrives) or True to False (blade leaves).
5. Pair each arrival with the next departure and discard incomplete pulses at either end.

The percentiles only work if the beam is both blocked and unblocked for more than 5 % of the
recording, which is true for any ordinary fan. Room light and amplifier offsets raise both
levels by the same amount, so a threshold halfway between them still works.

Measuring at the half-way level also means the beam diameter enters symmetrically: each
edge is detected when the blade covers half of the beam.

> **🧠 In the lab:** this is threshold-crossing **spike detection**. In ephys the events are
> small compared with the noise, so the threshold is set from the noise rather than halfway to
> a known level. It is typically 4–5 × σ, with the noise σ estimated robustly as
> `median(|x|) / 0.6745` so the spikes themselves don't inflate it (Quiroga et al., 2004).
> `MIN_PULSE_S` plays the role of discarding events too short to be real, and the
> *hysteresis* idea at the end of the notebook is a common fix for noisy threshold crossings.

In [ ]:
def levels(pd):
    """Return (dark, light) levels of one recording."""
    return np.percentile(pd, 5), np.percentile(pd, 95)


def find_pulses(signal, threshold, rate, min_width_s=MIN_PULSE_S):
    """Find intervals where signal < threshold.

    Returns (starts, widths) in seconds. Only complete pulses are returned.
    """
    blocked = (signal < threshold).astype(int)
    edges = np.diff(blocked)
    starts = np.flatnonzero(edges == 1) + 1
    ends = np.flatnonzero(edges == -1) + 1

    if starts.size == 0 or ends.size == 0:
        return np.array([]), np.array([])

    ends = ends[ends > starts[0]]          # drop a pulse that was already underway
    n = min(starts.size, ends.size)        # drop a pulse still underway at the end
    starts, ends = starts[:n], ends[:n]

    widths = (ends - starts) / rate
    keep = widths >= min_width_s           # crude de-glitching
    return starts[keep] / rate, widths[keep]

In [ ]:
dark, light = levels(pd)
threshold = (dark + light) / 2
print(f"dark = {dark:.3f} V, light = {light:.3f} V, threshold = {threshold:.3f} V")

if light - dark < 0.1:
    print("⚠️  Very little contrast between light and dark: is the fan spinning? Check the alignment with live_view()")

starts, widths = find_pulses(pd, threshold, RATE)
print(f"Found {len(widths)} pulses")

Check the detection by eye: the shaded regions should line up with the dips.

In [ ]:
show = t < 0.1

plt.figure(figsize=(10, 3))
plt.plot(t[show] * 1000, pd[show], lw=0.8)
plt.axhline(threshold, color="k", ls=":", label="threshold")
for s, w in zip(starts, widths):
    if s < 0.1:
        plt.axvspan(s * 1000, (s + w) * 1000, color="orange", alpha=0.3)
plt.xlabel("Time (ms)")
plt.ylabel("Photodiode (V)")
plt.legend(loc="lower right")
plt.title("Detected blade pulses (first 100 ms)")
plt.show()

---
## 4. Pulse width → blade width

* **Blade period** T: the time between successive pulse starts.
* **Rotation frequency** f = 1 / (N·T).
* **Blade speed at the beam** v = 2πrf.
* **Blade width** = v·τ − beam diameter.

We compute a width for every pulse. Taking the median is robust to the odd bad detection.

In [ ]:
def blade_widths(starts, widths, n_blades=N_BLADES, radius_mm=BEAM_RADIUS_MM,
                 beam_mm=BEAM_DIAMETER_MM):
    """Per-pulse blade width (mm) and the rotation frequency (Hz)."""
    if len(starts) < 2:
        raise ValueError("Need at least 2 pulses. Is the fan spinning and the beam aligned?")
    blade_period = np.median(np.diff(starts))
    rot_freq = 1 / (n_blades * blade_period)
    speed_mm_s = 2 * np.pi * radius_mm * rot_freq
    return speed_mm_s * widths - beam_mm, rot_freq


w_mm, rot_freq = blade_widths(starts, widths)

print(f"Rotation:      {rot_freq:.2f} Hz  ({rot_freq * 60:.0f} rpm)")
print(f"Pulse width:   {np.median(widths) * 1e3:.3f} ms (median)")
print(f"Blade width:   {np.median(w_mm):.2f} mm  "
      f"(median, spread ±{np.std(w_mm):.2f} mm over {len(w_mm)} pulses)")
if SIMULATE:
    print(f"[sim] true blade width was {SIM_TRUE_BLADE_MM} mm")

> **🧠 In the lab:** `speed × time = distance` is the whole calculation behind a scanning
> microscope's pixel size. For example, a resonant scanner at 8 kHz that sweeps 500 µm in roughly
> half of each 125 µs period, sampled into 512 pixels, gives about 1 µm per pixel and about 0.1 µs per
> pixel. In ephys, the same per-event numbers become **spike width** and **firing rate**
> (1 / inter-spike interval). Spike width is often used to tell narrow-spiking, putative
> fast-spiking interneurons apart from broad-spiking, putative pyramidal cells.

### Each blade separately

Pulse *k* comes from blade *k mod N*, so grouping by that index shows whether the blades
differ from one another. Which group is "blade 1" is arbitrary, because we don't know which blade passed first.

In [ ]:
blade_index = np.arange(len(w_mm)) % N_BLADES
per_blade = [w_mm[blade_index == b] for b in range(N_BLADES)]

plt.figure(figsize=(6, 3))
plt.boxplot(per_blade)
plt.xlabel("Blade (arbitrary numbering)")
plt.ylabel("Width (mm)")
plt.title("Width of each blade")
plt.show()

If one blade's box is clearly offset from the others, check that `N_BLADES` is correct.
If it's wrong, the per-blade groups mix different blades together and the rotation frequency is off by a factor.

> **🧠 In the lab:** sorting events by "which blade made them" is a very simple version of
> **spike sorting**, i.e. working out which neuron each spike on an electrode came from. Here it's
> easy because the blades arrive in a fixed order. With real neurons you group spikes by waveform
> shape instead, and getting the number of groups wrong causes the same kind of mixing.

---
## 5. Checking the measurement

We can't change the fan's speed, but we can still test the measurement in two ways.

### 5a. Is it repeatable?

Record several times in a row. A fan on its own supply doesn't hold a perfectly constant
speed: it drifts a little with warm-up, supply voltage and air flow, so the rpm will differ
slightly between recordings. Because the width depends only on τ/T, it **shouldn't** follow
those changes. If the width moves with the rpm, the edges are being measured wrongly.

In [ ]:
def analyse(pd, rate):
    """Blade widths (mm) and rotation frequency (Hz) from one recording."""
    dark, light = levels(pd)
    starts, widths = find_pulses(pd, (dark + light) / 2, rate)
    return blade_widths(starts, widths)


N_REPEATS = 5
repeats = []

for i in range(N_REPEATS):
    _, pd_i = record()
    w_i, f_i = analyse(pd_i, RATE)
    repeats.append((f_i * 60, np.median(w_i), np.std(w_i)))
    print(f"Recording {i + 1}: {f_i * 60:6.0f} rpm, width {np.median(w_i):6.2f} ± {np.std(w_i):.2f} mm")

rpm, width, spread = np.array(repeats).T
print(f"\nBlade width over {N_REPEATS} recordings: {width.mean():.2f} ± {width.std():.2f} mm")

### 5b. Is the sample rate high enough?

Each pulse edge can only be placed to the nearest sample, so the width of each pulse is
uncertain by about ±1 sample. At a low sample rate, a pulse is only a few samples long and
that error becomes large. Record at several rates: the spread should shrink as the rate goes
up, and the median should level off. Keep every rate at or below your device's maximum.

> **🧠 In the lab:** this is the same choice as an ephys sampling rate. A spike is about 1 ms
> long, so you need ≥20 kHz to see its shape and measure its width, even though a few kHz
> would be enough to just count spikes. In a microscope, the time per pixel sets how many PMT
> samples go into each pixel. A detector or amplifier that is too slow (low bandwidth) has a
> similar effect: it rounds off the blade edges, and the result starts to depend on speed.

In [ ]:
SAMPLE_RATES = [1_000, 2_000, 5_000, 10_000, 20_000]   # don't go above your device's maximum
by_rate = []

for r in SAMPLE_RATES:
    _, pd_r = record(rate=r)
    w_r, _ = analyse(pd_r, r)
    by_rate.append((np.median(w_r), np.std(w_r)))
    print(f"{r:6d} samples/s: width {np.median(w_r):6.2f} ± {np.std(w_r):.2f} mm")

rate_width, rate_spread = np.array(by_rate).T

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.errorbar(rpm, width, yerr=spread, fmt="o", capsize=4)
ax1.axhline(width.mean(), color="k", ls=":")
ax1.set_xlabel("Speed (rpm)")
ax1.set_ylabel("Blade width (mm)")
ax1.set_title("5a: repeated recordings (should be flat)")

ax2.errorbar(SAMPLE_RATES, rate_width, yerr=rate_spread, fmt="o-", capsize=4)
ax2.set_xscale("log")
ax2.set_xlabel("Sample rate (samples/s)")
ax2.set_ylabel("Blade width (mm)")
ax2.set_title("5b: width vs sample rate")
plt.tight_layout()
plt.show()

---
## 6. Finishing up

The DAQ doesn't output anything in this notebook, so there is nothing to reset in software.
Switch the laser and the fan off **at their own power supplies**, and wait for the fan to
stop before touching the setup.

## Things to try

* **Measure with a ruler** and compare. Fan blades are angled (pitched), so what the beam measures
  is the blade's width along the direction of rotation at radius r, not along its surface.
* **Move the beam to a different radius.** Many blades get wider towards the tip.
* **Better edge timing:** interpolate between the two samples either side of each threshold
  crossing, instead of using whole samples. This matters when τ is only a few samples long.
  Repeat 5b and see whether the low sample rates improve.
* **Hysteresis:** use two thresholds (e.g. 40 % and 60 %) instead of one, so noise near the
  threshold can't create extra edges. This is more robust than `MIN_PULSE_S`.
* **How steady is the fan?** Record for 10 s or more, compute the blade period T for each pair of
  pulses, and plot it against time. This is how you would check a resonant scanner's
  frequency for drift.
* **Measure the beam diameter** by moving a razor blade across the beam on a translation stage
  while watching `live_view()`. This "knife-edge" method is also used to measure laser spot sizes
  on real microscopes.
* **Make an image:** record for one full rotation and reshape the signal into rows, one per
  rotation, over several rotations. You've built a (very slow) scanning imager. Compare it with
  how a microscope reshapes the PMT signal into a frame.